# P8: GRU va LSTM ni taqqoslash
**Mavzu:** `nn.LSTM`/`nn.GRU` (num_layers=2) klassifikator, GRU-vs-LSTM taqqoslash, vanishing gradient
**Kun:** 9-kun amaliyoti — 29-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L8 — GRU va LSTM (`d08_gru_lstm.pdf`)
**Kapstone modul:** m08 `GRULSTMClassifier`
**Ma'lumot:** uz_news_full (online). Offline: `d03_checkpoints/uz_sentiment_mini.txt` (ijobiy/salbiy).

## Bugungi maqsadlar (ma'ruza [C] dan)
1. LSTM forget gate qadamini ($\sigma(1.5)=0.818$) qo'lda hisoblaymiz.
2. LSTM va GRU klassifikatorlarini quramiz va o'qitamiz.
3. Ikkalasini bir xil vazifada taqqoslaymiz (`compare_report`).
4. Uzun matnlarda vanishing gradient ta'sirini tahlil qilamiz.

## Vaqt taqsimoti
| Bo'lim | Vaqt | Mazmun |
|---|---|---|
| §1 Muhit | 4 daq | seedlar, OFFLINE_FALLBACK, HAS_TORCH |
| §2 Yaxlit natija | 8 daq | tayyor compare_report (GRU vs LSTM) |
| §3 Periferiya (PRIMM) | 17 daq | m06 pretrained Embedding; training loop, learning curve |
| §4 Yadro | 35 daq | forget gate (qulflangan), LSTM/GRU qurish, taqqoslash, vanishing |
| §5 Loyihaga ulash | 13 daq | m08 moduli, import test, git |
| §6 Tadqiqot + yakun | 7 daq | GRU vs LSTM tanlovi |

## 1. Muhit tekshiruvi
Seedlar, offline bayroq, m01 yo'li, `torch` bayrog'i. GPU shart emas (CPU yetarli).

In [ ]:
# Kaggle: torch oldindan o'rnatilgan (GPU bo'lsa tezlashadi, lekin talab emas)
import os, sys, random, math, tempfile
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True

try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
    print("torch mavjud:", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("torch yo'q — toza-numpy zaxira (reservoir + LogReg) ishlatiladi.")

MODULES = os.path.join("..", "..", "capstone", "modules")
if not os.path.isdir(MODULES):
    MODULES = os.path.join("capstone", "modules")
sys.path.insert(0, MODULES)

from m01_text_preprocessor import TextPreprocessor
print("numpy:", np.__version__, "| OFFLINE_FALLBACK =", OFFLINE_FALLBACK)


## 2. Yaxlit natija (avval manzil)
Tayyor `GRULSTMClassifier` sentiment ma'lumotida LSTM va GRU ni **bir xil vazifada**
o'qitib taqqoslaydi (`compare_report`). Tafsilotlarni keyin ochamiz.

In [ ]:
from m08_gru_lstm_classifier import GRULSTMClassifier

SENT = os.path.join("..", "..", "course", "practices", "d03_checkpoints", "uz_sentiment_mini.txt")
if not os.path.exists(SENT):
    SENT = os.path.join("course", "practices", "d03_checkpoints", "uz_sentiment_mini.txt")

def yukla_sentiment(path):
    texts, labels = [], []
    for line in open(path, encoding="utf-8"):
        if not line.strip():
            continue
        rating, text = line.rstrip("\n").split("\t", 1)
        r = int(rating)
        if r >= 4:   texts.append(text); labels.append("ijobiy")
        elif r <= 2: texts.append(text); labels.append("salbiy")
    return texts, labels

texts, labels = yukla_sentiment(SENT)
print("Misollar:", len(texts), "| ijobiy:", labels.count("ijobiy"), "salbiy:", labels.count("salbiy"))

clf_demo = GRULSTMClassifier()
clf_demo.fit(texts, labels, arch="lstm", epochs=8, hidden_size=32, num_layers=2, lr=0.01)
print("compare_report (GRU vs LSTM):")
for arch, m in clf_demo.compare_report().items():
    print(" ", arch, m)


## 3. Tayyor kod bloki — pretrained Embedding va o'qitish halqasi (PRIMM)
Bu bo'lim periferiya. RNN kirishini m06 (CustomWord2Vec) embeddinglari bilan
\textbf{boshlash} (pretrained) ko'pincha tasodifiy initdan yaxshiroq. Quyida m06 dan
embedding matritsa quramiz.

**Bashorat qiling:** pretrained embedding matritsa shakli qanday bo'ladi? (Lug'at hajmi × o'lcham.)

In [ ]:
# PERIFERIYA — m06 dan pretrained embedding matritsasi quramiz.
from m06_custom_word2vec import CustomWord2Vec

pre = TextPreprocessor()
sents = [pre.preprocess(t) for t in texts]
w2v = CustomWord2Vec()
w2v.train(sents, vector_size=32, window=2, min_count=1, epochs=10)

# lug'at -> embedding matritsa (0-qator PAD uchun nol)
vocab = sorted({t for s in sents for t in s})
w2i = {w: i + 1 for i, w in enumerate(vocab)}
emb_matrix = np.zeros((len(w2i) + 1, 32), dtype=np.float32)
for w, i in w2i.items():
    emb_matrix[i] = w2v.embed(w)
print("Pretrained embedding matritsa shakli:", emb_matrix.shape)
# Kaggle: nn.Embedding.from_pretrained(torch.tensor(emb_matrix))


**Tekshiring:**
1. `emb_matrix[0]` nega nol? (PAD tokeni.)
2. Pretrained embedding nega tasodifiy initdan ustun bo'lishi mumkin?
3. `num_layers=2` ikki qatlamli RNN nimani anglatadi?

**O'zgartiring:** `vector_size` ni 16 ga o'zgartirib, matritsa shaklini kuzating.

In [ ]:
# PERIFERIYA — o'qitish halqasi va learning curve (illyustrativ).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def learning_curve(history):
    """history: {'loss': [...], 'val_f1': [...]} -> grafik (Agg)."""
    fig, ax = plt.subplots(1, 2, figsize=(7, 2.5))
    ax[0].plot(history["loss"]); ax[0].set_title("Loss")
    ax[1].plot(history["val_f1"]); ax[1].set_title("Val F1")
    plt.tight_layout(); plt.close(fig)
    return "grafik tayyor"

print(learning_curve({"loss": [0.69, 0.4, 0.2], "val_f1": [0.5, 0.7, 0.85]}))


### Checkpoint A
Orqada qolsangiz — ma'lumotni qaytadan yuklaydi.

In [ ]:
if OFFLINE_FALLBACK or "texts" not in dir():
    texts, labels = yukla_sentiment(SENT)
print("Checkpoint: ma'lumot tayyor —", len(texts), "misol")


## 4. Asosiy mavzu — LSTM va GRU (so'nuvchi tayanch)
**Namuna → Birgalikda → Mustaqil.** Avval ma'ruzaning forget gate misolini takrorlaymiz.

### 4A. Namuna (men qilaman): LSTM forget gate
Ma'ruza L8 [I3]: $W_f=1$, $b_f=0$, $h_{t-1}=0.5$, $x_t=1.0$. Forget gate ---
$f_t=\sigma(W_f h_{t-1}+W_f x_t+b_f)$. Toza-numpy (torch shart emas).

In [ ]:
# LSTM forget gate (toza-numpy) — L8 [I3]
Wf, bf, h_prev, x = 1.0, 0.0, 0.5, 1.0
f_t = 1 / (1 + np.exp(-(Wf*h_prev + Wf*x + bf)))    # sigma(1.5)
print("forget gate f_t =", round(f_t, 4))

assert abs(f_t - 0.818) < 1e-3      # Ma'ruza L8 [I3]-slayd bilan solishtiring
print("To'g'ri! f_t = sigma(1.5) = 0.818 (o'tmish ko'p saqlanadi)")


### 4B. Birgalikda (biz qilamiz): LSTM klassifikatorini o'qitamiz
`GRULSTMClassifier` ichida `nn.LSTM(num_layers=2)` (Kaggle) yoki reservoir (offline).
CPU uchun kichik hidden va kam epoch (to'liq miqyos: hidden=128, epochs=10 GPU'da).

In [ ]:
clf = GRULSTMClassifier()
# === SIZNING KODINGIZ (taxminan 1 qator) ===
# clf ni LSTM arxitekturasi bilan o'qiting: arch='lstm', epochs=8, hidden_size=32, num_layers=2, lr=0.01
clf.fit(...)


In [ ]:
assert len(clf._w2i) > 0, "Lug'at bo'sh — fit() chaqirilganini tekshiring."
assert clf.predict("juda zor mahsulot rahmat") in {"ijobiy", "salbiy"}, \
    "predict() 'ijobiy' yoki 'salbiy' qaytarishi kerak."
print("Ajoyib! LSTM o'qitildi — lug'at:", len(clf._w2i), "so'z; yorliqlar:", clf._labels)


### 4C. Birgalikda (biz qilamiz): GRU va LSTM ni taqqoslaymiz
`compare_report()` ikkala arxitekturani bir xil vazifada o'qitib, F1, aniqlik va
inference vaqtini qaytaradi.

In [ ]:
# === SIZNING KODINGIZ (taxminan 1 qator) ===
# compare_report() ni chaqiring
report = ...
for arch, m in report.items():
    print(arch, "->", m)


In [ ]:
assert set(report.keys()) == {"lstm", "gru"}, "Hisobotda 'lstm' va 'gru' bo'lishi kerak."
assert all("f1" in report[a] and "inference_time" in report[a] for a in report), \
    "Har arxitektura uchun f1 va inference_time bo'lishi kerak."
print("To'g'ri! compare_report GRU va LSTM ni taqqosladi.")


### 4D. Mustaqil (siz qilasiz): vanishing gradient va saqlash
Uzun va qisqa ketma-ketlikda gradient oqimini taqqoslang (L8 [I2]): oddiy RNN bo'g'ini
$\approx 0.42$, LSTM cell forget $\approx 0.9$. Modelni saqlang va qayta yuklang. Tayanch yo'q.

In [ ]:
# === SIZNING KODINGIZ (taxminan 8 qator) ===
# 1) N=3 va N=15 uchun RNN (0.42**N) va LSTM (0.9**N) gradientini chop eting
# 2) assert: 0.9**15 > 0.42**15 (LSTM uzoqda gradientni yaxshi saqlaydi)
# 3) clf ni saqlang, yangi obyektga yuklang
# 4) assert: yuklangan model bir xil bashorat beradi
...


## 5. Loyihaga ulash — m08 `GRULSTMClassifier`
Bugungi ish `capstone/modules/m08_gru_lstm_classifier.py` da (shartnoma
`capstone/contracts.py`). Torch-ixtiyoriy: Kaggle'da `nn.LSTM`/`nn.GRU`, offline'da reservoir.

In [ ]:
from m08_gru_lstm_classifier import GRULSTMClassifier

m = GRULSTMClassifier()
for metod in ["fit", "predict", "compare_report", "save", "load"]:
    assert callable(getattr(m, metod, None)), f"m08.{metod} mavjud emas!"
print("m08 GRULSTMClassifier shartnomaga mos: fit / predict / compare_report / save / load")


```bash
git add capstone/modules/m08_gru_lstm_classifier.py
git commit -m "P8: m08 GRULSTMClassifier — GRU vs LSTM"
git push
```

## 6. Tadqiqot savoli + yakun
**Mini-tajriba:** `compare_report` natijasida GRU va LSTM ning F1 va inference vaqtini
solishtiring. Kichik ma'lumotda qaysi biri yaxshiroq?

In [ ]:
rep = clf.compare_report()
tezroq = min(rep, key=lambda a: rep[a]["inference_time"])
aniqroq = max(rep, key=lambda a: rep[a]["f1"])
print("Tezroq:", tezroq, "| Aniqroq (F1):", aniqroq)
print("Eslatma: kichik korpusda GRU (kamroq parametr) ko'pincha barqarorroq.")


**Savol (o'ylab ko'ring):** o'zbekcha uzun qo'shimcha zanjiri ("o'rganilganlardan")
va SOV (kesim oxirida) uchun GRU yoki LSTM — qaysi biri afzal? Nega? Kichik o'zbek
korpusida parametr soni qanday rol o'ynaydi?

**Chiqish chiptasi:** _Bugun eng tushunarsiz qolgan narsa nima?_